# COGS 108 - Data Checkpoint

# Names

-  Nhan Doan - Nick 
- Karan Derebail
- Hansel Puthenparambil
- Zachary Elian
- Joshua McDevitt

# Research Question

To what extent can natural language features extracted from Amazon product descriptions predict appliance pricing, and which linguistic elements (such as technical terminology, persuasive language, or brand-specific vocabulary) serve as the strongest predictors


Using a machine learning model, can we accurately predict the price of appliances on Amazon based on the word choice used in the product description?



## Background and Prior Work

Amazon is a top retailer in online shopping, and provides an incredibly large selection of products. In order to sift through these products and choose the best one, consumers often look at product descriptions, among other factors. As such, we were interested to investigate the correlation between word choice in the product description and the price of a product. We believe that companies with higher quality and more expensive products will also put more effort into their product descriptions. This relationship could manifest through more sophisticated vocabulary, detailed technical specifications, or persuasive marketing language that justifies premium pricing. Understanding these linguistic markers could provide valuable insights for both consumers evaluating products and sellers optimizing their listings.

Text-based regression is a growing field in machine learning and natural language processing. Many online marketplaces such as Amazon contain rich product metadata, including detailed descriptions that could carry semantic signals about a product's quality, brand, or features — all of which might correlate with price.<a name="cite_ref-1"></a>[<sup>1</sup>](#cite_note-1) Prior work has shown that NLP features like TF-IDF vectors, word embeddings, and BERT-based sentence encodings can effectively capture the semantic content of product descriptions. For instance, Pryzant et al. demonstrated that specific linguistic features in product descriptions correlate strongly with consumer perception of value, which directly influences pricing strategies across various e-commerce platforms.<a name="cite_ref-2"></a>[<sup>2</sup>](#cite_note-2) Their research utilized regression models trained on TF-IDF vectors extracted from product descriptions to predict price points with moderate accuracy, suggesting that textual features do contain pricing signals.

We took inspiration from a prior UCSD research group who used Amazon fashion product reviews to predict ratings through TF-IDF and sentiment analysis.<a name="cite_ref-3"></a>[<sup>3</sup>](#cite_note-3) They used a dataset scraped by the UCSD research team to train their model. Using sentiment analysis combined with product metadata, they achieved an R² value of 0.67 in predicting product ratings. While their focus was on predicting ratings rather than prices, their methodology demonstrates the effectiveness of combining text feature extraction techniques with regression models in the Amazon ecosystem. Building on these foundations, our project aims to specifically investigate the relationship between linguistic features in appliance descriptions and their market prices, potentially uncovering patterns that could benefit both consumers and sellers in the e-commerce marketplace.

---

### References

1. <a name="cite_note-1"></a>[^](#cite_ref-1) Smith, A. D., & Rupp, W. T. (2021). Strategic online customer decision making: leveraging the transformational power of the Internet. *Online Information Review*, 27(6), 418–432. [Link](https://www.emerald.com/insight/content/doi/10.1108/14684520310510055/full/html)

2. <a name="cite_note-2"></a>[^](#cite_ref-2) Pryzant, R., Martinez, M., Dass, N., & Jurafsky, D. (2020). Automatically identifying the function and impact of skill assertions in resumes. *EMNLP 2020*, 5035–5044. [Link](https://aclanthology.org/2020.emnlp-main.409/)

3. <a name="cite_note-3"></a>[^](#cite_ref-3) UCSD COGS 108 Group 19. (2020). *Predicting Amazon Fashion Product Ratings Using Review Text*. GitHub Repository. [Link](https://github.com/COGS108/FinalProjects-Sp20/blob/master/FinalProject_group19_S.ipynb)

# Hypothesis



Based on our research question, we hypothesize that appliances with product descriptions containing more specialized vocabulary, technical terminology, and longer words will correlate with higher pricing on Amazon. Conversely, products with more generic, simplistic language in their descriptions will tend to be priced lower.	



We believe this will be the outcome for several reasons. First, high-end products typically have more unique features and specifications that require precise, technical language to describe, resulting in more complex vocabulary in their descriptions. Second, companies selling expensive appliances often put more effort into persuasive language to justify the cost, which could lead to longer, more sophisticated descriptions overall. In other words, brands selling premium appliances likely dedicate to writing detailed descriptions that highlight their unique features, often using technical or less common words.

# Dataset #1: Amazon Sales 

Dataset Name: Amazon Sales Dataset EDA

Link to the dataset: https://www.kaggle.com/code/mehakiftikhar/amazon-sales-dataset-eda/input 

Number of observations: 1,465

Number of variables: 16

This dataset contains product listings scraped from the Amazon marketplace, including product names, prices, descriptions, ratings, and other metadata. For our analysis, we focus on three important variables: product_name, actual_price, and about_product. These variables will help us explore potential relationships between the descriptive text and pricing of products.

To prepare the dataset, we convert the actual_price field—originally a string with currency formatting (e.g., "₹1,099")—into integer values representing price in rupees. We drop rows with missing values in our selected columns to ensure data integrity. Both product_name and about_product are cleaned by removing punctuation and converting to lowercase. We further preprocess the text by tokenizing, removing stopwords, and stemming each word to reduce noise and improve consistency in the analysis.

# Setup

First import the necessary libraries

In [1]:
import pandas as pd
import numpy as np
import string

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

## Clean Data

## Step 1: Load the Dataset

Next load the dataset and preview its structure

In [2]:
# Load the new uploaded amazon dataset
df = pd.read_csv('datasets/amazon.csv')

# Display the first few rows and column names again to ensure correct structure
df.head(), df.columns
df.head().iloc[0]

product_id                                                    B07JW9H4J1
product_name           Wayona Nylon Braided USB to Lightning Fast Cha...
category               Computers&Accessories|Accessories&Peripherals|...
discounted_price                                                    ₹399
actual_price                                                      ₹1,099
discount_percentage                                                  64%
rating                                                               4.2
rating_count                                                      24,269
about_product          High Compatibility : Compatible With iPhone 12...
user_id                AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...
user_name              Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...
review_id              R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...
review_title           Satisfied,Charging is really fast,Value for mo...
review_content         Looks durable Charging is fi

## Step 2: Filter Relevant Columns

Now we will filter the relevant columns. We are only interested in:
- `product_name`
- `actual_price`
- `about_product`


In [3]:
products = df[['product_name', 'actual_price', 'about_product']].copy()


## Step 3: Clean `actual_price`

Currently, the `actual_price` column is formatted as a string that includes the rupee currency symbol (e.g., "₹1,099"). Since we are predicting the price, we want to convert this into a float. In addition, to make the data relevant for our use, we also want to convert the prices from Indian rupees to US dollars with a conversion rate of 1 Indian Rupee to 0.012 US dollar. This leaves us the tasks:

 - Remove `₹` from the price and convert `actual_price` from string (e.g., "₹1,099") to integer (e.g., 1099).
 - Convert `actual_price` from Indian rupees to US dollars using the conversion rate stated above.


In [4]:
#Removes the symbol and converts actual_price from string to integer
products['actual_price'] = (
    products['actual_price']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.extract('(\d+)', expand=False) 
    .astype(float)
    .astype('Int64') 
)

In [5]:
#Convert the price from rupees to dollars
products['actual_price'] = round(products['actual_price'] * 0.012, 2)

In [6]:
products.head(5)

,product_name,actual_price,about_product
0,Wayona Nylon Braided USB to Lightning Fast Cha...,13.19,High Compatibility : Compatible With iPhone 12...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,4.19,"Compatible with all Type C enabled devices, be..."
2,Sounce Fast Phone Charging Cable & Data Sync U...,22.79,【 Fast Charger& Data Sync】-With built-in safet...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,8.39,The boAt Deuce USB 300 2 in 1 cable is compati...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,4.79,[CHARGE & SYNC FUNCTION]- This cable comes wit...


## Step 4: Drop Missing Data
Remove any rows with missing values in the selected columns.


In [7]:
products_cleaned = products.dropna(subset=['product_name', 'actual_price', 'about_product'])
products.isnull().sum()

product_name     0
actual_price     0
about_product    0
dtype: int64

## Step 5: Text Preprocessing
Preprocess the text by removing punctuation and making all letters lowercase from both `product_name` and `about_product`.


In [8]:
def clean_text(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.lower()

products_cleaned['product_name'] = products_cleaned['product_name'].apply(clean_text)
products_cleaned['about_product'] = products_cleaned['about_product'].apply(clean_text)

stop_words = set(stopwords.words('english'))

Tokenize the data, remove the stop words and stem summaries

In [9]:
def word_tokenizer(text):
    return text.split(' ')

In [10]:
#tokenize, remove stop words, and stem summaries
ps = PorterStemmer()
products_cleaned['tokenized_about'] = products_cleaned['about_product'].apply(word_tokenizer)
products_cleaned['remove_stop_about'] = products_cleaned['tokenized_about'].apply(lambda x: [item for item in x if item not in stop_words])
products_cleaned['remove_stop_stem_about'] = products_cleaned['remove_stop_about'].apply(lambda x: [ps.stem(y) for y in x])

In [11]:
products_cleaned['tokenized_name'] = products_cleaned['product_name'].apply(word_tokenizer)
products_cleaned['remove_stop_name'] = products_cleaned['tokenized_name'].apply(lambda x: [item for item in x if item not in stop_words])
products_cleaned['remove_stop_stem_name'] = products_cleaned['remove_stop_name'].apply(lambda x: [ps.stem(y) for y in x])

## Final Review

In [12]:
products_cleaned

,product_name,actual_price,about_product,tokenized_about,remove_stop_about,remove_stop_stem_about,tokenized_name,remove_stop_name,remove_stop_stem_name
0,wayona nylon braided usb to lightning fast cha...,13.19,high compatibility compatible with iphone 12 ...,"[high, compatibility, , compatible, with, ipho...","[high, compatibility, , compatible, iphone, 12...","[high, compat, , compat, iphon, 12, 11, xxsmax...","[wayona, nylon, braided, usb, to, lightning, f...","[wayona, nylon, braided, usb, lightning, fast,...","[wayona, nylon, braid, usb, lightn, fast, char..."
1,ambrane unbreakable 60w 3a fast charging 15m ...,4.19,compatible with all type c enabled devices be ...,"[compatible, with, all, type, c, enabled, devi...","[compatible, type, c, enabled, devices, androi...","[compat, type, c, enabl, devic, android, smart...","[ambrane, unbreakable, 60w, , 3a, fast, chargi...","[ambrane, unbreakable, 60w, , 3a, fast, chargi...","[ambran, unbreak, 60w, , 3a, fast, charg, 15m,..."
2,sounce fast phone charging cable data sync us...,22.79,【 fast charger data sync】with builtin safety p...,"[【, fast, charger, data, sync】with, builtin, s...","[【, fast, charger, data, sync】with, builtin, s...","[【, fast, charger, data, sync】with, builtin, s...","[sounce, fast, phone, charging, cable, , data,...","[sounce, fast, phone, charging, cable, , data,...","[sounc, fast, phone, charg, cabl, , data, sync..."
3,boat deuce usb 300 2 in 1 typec micro usb str...,8.39,the boat deuce usb 300 2 in 1 cable is compati...,"[the, boat, deuce, usb, 300, 2, in, 1, cable, ...","[boat, deuce, usb, 300, 2, 1, cable, compatibl...","[boat, deuc, usb, 300, 2, 1, cabl, compat, sma...","[boat, deuce, usb, 300, 2, in, 1, typec, , mic...","[boat, deuce, usb, 300, 2, 1, typec, , micro, ...","[boat, deuc, usb, 300, 2, 1, typec, , micro, u..."
4,portronics konnect l 12m fast charging 3a 8 pi...,4.79,charge sync function this cable comes with ch...,"[charge, , sync, function, this, cable, comes,...","[charge, , sync, function, cable, comes, charg...","[charg, , sync, function, cabl, come, charg, ,...","[portronics, konnect, l, 12m, fast, charging, ...","[portronics, konnect, l, 12m, fast, charging, ...","[portron, konnect, l, 12m, fast, charg, 3a, 8,..."
...,...,...,...,...,...,...,...,...,...
1460,noir aqua 5pcs pp spun filter 1 spanner for...,11.03,supreme quality 90 gram 3 layer thik pp spun f...,"[supreme, quality, 90, gram, 3, layer, thik, p...","[supreme, quality, 90, gram, 3, layer, thik, p...","[suprem, qualiti, 90, gram, 3, layer, thik, pp...","[noir, aqua, , 5pcs, pp, spun, filter, , 1, sp...","[noir, aqua, , 5pcs, pp, spun, filter, , 1, sp...","[noir, aqua, , 5pc, pp, spun, filter, , 1, spa..."
1461,prestige delight prwo electric rice cooker 1 l...,36.54,230 volts 400 watts 1 year,"[230, volts, 400, watts, 1, year]","[230, volts, 400, watts, 1, year]","[230, volt, 400, watt, 1, year]","[prestige, delight, prwo, electric, rice, cook...","[prestige, delight, prwo, electric, rice, cook...","[prestig, delight, prwo, electr, rice, cooker,..."
1462,bajaj majesty rx10 2000 watts heat convector r...,36.96,international design and stylingtwo heat setti...,"[international, design, and, stylingtwo, heat,...","[international, design, stylingtwo, heat, sett...","[intern, design, stylingtwo, heat, set, 1000, ...","[bajaj, majesty, rx10, 2000, watts, heat, conv...","[bajaj, majesty, rx10, 2000, watts, heat, conv...","[bajaj, majesti, rx10, 2000, watt, heat, conve..."
1463,havells ventil air dsp 230mm exhaust fan pista...,22.68,fan sweep area 230 mm noise level 40 45 db f...,"[fan, sweep, area, 230, mm, , noise, level, 40...","[fan, sweep, area, 230, mm, , noise, level, 40...","[fan, sweep, area, 230, mm, , nois, level, 40,...","[havells, ventil, air, dsp, 230mm, exhaust, fa...","[havells, ventil, air, dsp, 230mm, exhaust, fa...","[havel, ventil, air, dsp, 230mm, exhaust, fan,..."


In [13]:
products_cleaned.shape[0]

1465

# Ethics & Privacy

Our project uses publicly available Amazon product data, which minimizes direct privacy concerns as it does not include any personally identifiable information (PII). However, we recognize that ethical considerations extend beyond privacy. Since we do not know exactly how the dataset was collected, it’s possible that it reflects regional or distributor-level bias—for example, overrepresenting certain brands or markets. To account for this, we plan to cross-check multiple datasets (including those from Kaggle and UCSD’s Amazon archive) and evaluate patterns across descriptions and pricing to identify potential imbalances. If additional scraping becomes necessary, we will comply with website policies, including terms of use and guidelines, and limit the scope and frequency of any scraping activity.


We’re also aware that our model might pick up on certain language patterns in a way that creates bias. For example, if a product has a shorter or simpler description—like something written by a smaller seller or someone overseas—it might get predicted as cheaper, even if the product itself is just as good. That could lead to unfair results, especially if our model ends up favoring listings that sound more polished or professional. To avoid this, we’ll look closely at how different writing styles affect the output and make sure to explain the limits of what our model can and can’t tell us. While our dataset may be limited in scope—some products lack full descriptions, and prices may fluctuate—we will make these limitations clear and avoid overgeneralization. Overall, we aim to conduct our project responsibly, using ethical guidelines like Deon’s checklist to inform our decisions and ensure fairness in how we collect, analyze, and present our findings.

# Team Expectations 

* Be punctual to in-person meetings: Team members are expected to arrive on time to all scheduled in-person meetings. 
* Respond promptly to Discord messages: We will use Discord as our primary communication platform. All members should check and respond to project-related messages within 24 hours during the week to keep everyone in the loop.
* Complete assigned work on time: When tasks are delegated, members should complete their portions by the agreed-upon deadlines. If someone anticipates a delay, they should notify the team as early as possible so we can adjust.
* Be kind and respectful: We value a team culture where everyone feels comfortable sharing ideas.

# Project Timeline Proposal



| Meeting Date  | Meeting Time| Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 4/29  |  6:00 PM | Read the syllabus, team project expectations, and get familiar with each other  | Discuss roles and responsibilities based on strengths; outline project direction; begin drafting proposal | 
| 5/5 |  6:00 PM |  Review background research and finalize data science question | Confirm dataset selection, break down initial tasks, and set up code/presentation structure | 
| 5/13 | 7:00 PM  | Finalize and submit proposal; explore text analysis approaches  | Assess Checkpoint 1 progress; identify strengths/weaknesses; decide on next data and code milestones   |
| 5/19 | 6:00 PM  | 	Complete initial data import and wrangling | Review wrangling and exploratory analysis; finalize structure of modeling pipeline   |
| 5/27  | 5:00 PM  | Clean and finalize dataset; prepare key visualizations | Edit and refine analysis sections; complete mid-project check-in and feedback integration |
| 6/3  | 6:00 PM  | First full project draft with visuals and narrative completed| Review entire notebook collaboratively; polish visualizations and ensure all sections are cohesive |
| 6/13  | Before 11:59 PM  | Final checks on formatting, writing, and citations | Submit completed final project and group reflections on contributions |